In [0]:
USE CATALOG demo;
USE SCHEMA data_security;

In [0]:
CREATE OR REPLACE TABLE sales_secured (
    id INT,
    region STRING,
    email STRING,
    revenue INT
)

In [0]:
INSERT INTO sales_secured VALUES
    (1, 'UK', 'jane@sales.com', 100),
    (2, 'US', 'bob@sales.com', 200),
    (3, 'US', 'jane@sales.com', 300),
    (4, 'UK', 'bob@sales.com', 400),
    (5, 'US', 'jane@sales.com', 500),
    (6, 'UK', 'bob@sales.com', 600),
    (7, 'US', 'jane@sales.com', 700),
    (8, 'UK', 'bob@sales.com', 800),
    (9, 'US', 'jane@sales.com', 900),
    (10, 'UK', 'bob@sales.com', 1000)

In [0]:
SELECT * FROM sales_secured;

##### Step 4: Define Row Filer Function

In [0]:
CREATE OR REPLACE FUNCTION fn_filter_region (region STRING)
RETURN 
    is_account_group_member('admin-sg')
    OR (is_account_group_member('uk-sg') AND region = 'UK')
    OR (is_account_group_member('us-sg') AND region = 'US');

##### Step 5: Apply Row Filter to Table

In [0]:
ALTER TABLE sales_secured
SET ROW FILTER fn_filter_region ON (region);

In [0]:
DESC EXTENDED sales_secured;

In [0]:
SELECT * FROM sales_secured;

#### Step 6: Define Column Mask Function

In [0]:
CREATE OR REPLACE FUNCTION fn_mask_email(email STRING)
RETURN 
    CASE 
        WHEN is_account_group_member('admin-sg') THEN email
        ELSE concat(substr(email, 1, 1), '***@', split(email, '@')[1])
    END;

###### Step 7: Apply Column Mask

In [0]:
ALTER TABLE sales_secured
ALTER COLUMN email SET MASK fn_mask_email;

In [0]:
DESC EXTENDED sales_secured;

In [0]:
SELECT * FROM sales_secured;